In [2]:
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import time
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import random

# Download dataset
itsahmad_indoor_scenes_cvpr_2019_path = kagglehub.dataset_download('itsahmad/indoor-scenes-cvpr-2019')
print('Data source import complete.')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Setup paths
basePath = itsahmad_indoor_scenes_cvpr_2019_path
imgDir = os.path.join(basePath, "indoorCVPR_09", "Images")
trainLabelFile = os.path.join(basePath, "TrainImages.txt")
testLabelFile = os.path.join(basePath, "TestImages.txt")

# FIXED: Create proper train/validation split
def create_train_val_split(train_file, val_ratio=0.2):
    """Create a proper train/validation split from the training data"""
    with open(train_file, 'r') as f:
        train_lines = f.read().splitlines()
    
    class_to_images = defaultdict(list)
    for line in train_lines:
        class_name = line.split('/')[0]
        class_to_images[class_name].append(line)
    
    train_list, val_list = [], []
    for cls, images in class_to_images.items():
        random.shuffle(images)
        split_idx = int((1 - val_ratio) * len(images))
        train_list += images[:split_idx]
        val_list += images[split_idx:]
    
    return train_list, val_list

# Create proper split
train_list, val_list = create_train_val_split(trainLabelFile, val_ratio=0.2)
print(f"Train: {len(train_list)} images, Val: {len(val_list)} images")

class Indoor67Dataset(Dataset):
    def __init__(self, imgDir, image_list, transform=None):
        self.samples = []
        self.transform = transform
        
        for imgName in image_list:
            if imgName.strip():
                label = imgName.split('/')[0]
                fullPath = os.path.join(imgDir, imgName)
                if os.path.isfile(fullPath):
                    self.samples.append((fullPath, label))
        
        self.classes = sorted(set(lbl for _, lbl in self.samples))
        self.classToIdx = {cls: i for i, cls in enumerate(self.classes)}
        print(f"Loaded {len(self.samples)} samples with {len(self.classes)} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.classToIdx[label]

# FIXED: Better data augmentation and preprocessing
trainTransform = transforms.Compose([
    transforms.Resize((256, 256)),  # Resize first
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),  # More aggressive cropping
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),  # Add grayscale augmentation
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

valTransform = transforms.Compose([
    transforms.Resize((224, 224)),  # FIXED: No random crop for validation
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

testTransform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create datasets
trainDataset = Indoor67Dataset(imgDir, train_list, transform=trainTransform)
valDataset = Indoor67Dataset(imgDir, val_list, transform=valTransform)

# For test, read from file
with open(testLabelFile, 'r') as f:
    test_list = f.read().splitlines()
testDataset = Indoor67Dataset(imgDir, test_list, transform=testTransform)

# FIXED: Better batch size and data loading (fix multiprocessing issue)
trainLoader = DataLoader(trainDataset, batch_size=32, shuffle=True, num_workers=0)
valLoader = DataLoader(valDataset, batch_size=32, shuffle=False, num_workers=0)
testLoader = DataLoader(testDataset, batch_size=32, shuffle=False, num_workers=0)

# FIXED: Use pretrained ResNet50 instead of custom model (fix deprecation warning)
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
# Replace final layer for 67 classes
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_features, len(trainDataset.classes))
)
model = model.to(device)

# FIXED: Better loss function and optimizer
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

# FIXED: Better learning rate scheduler
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=0.003, 
    epochs=50, 
    steps_per_epoch=len(trainLoader),
    pct_start=0.1
)

def train_model(model, train_loader, val_loader, num_epochs=50, patience=10):
    train_losses, train_accs, val_losses, val_accs = [], [], [], []
    best_val_acc = 0.0
    patience_counter = 0
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # FIXED: Add gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()  # Step per batch for OneCycleLR
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_acc = correct / total
        train_loss = running_loss / len(train_loader)
        train_accs.append(train_acc)
        train_losses.append(train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_acc = val_correct / val_total
        val_loss = val_loss / len(val_loader)
        val_accs.append(val_acc)
        val_losses.append(val_loss)
        
        print(f'Epoch [{epoch+1}/{num_epochs}] - '
              f'Train Acc: {train_acc:.4f}, Loss: {train_loss:.4f} | '
              f'Val Acc: {val_acc:.4f}, Loss: {val_loss:.4f} | '
              f'LR: {scheduler.get_last_lr()[0]:.6f}')
        
        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            # Save best model
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    return train_losses, train_accs, val_losses, val_accs

# Train the model
print("Starting training...")
train_losses, train_accs, val_losses, val_accs = train_model(
    model, trainLoader, valLoader, num_epochs=50, patience=15
)

# Load best model for testing
model.load_state_dict(torch.load('best_model.pth'))

# Test evaluation
def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = correct / total
    return accuracy, all_predictions, all_labels

test_accuracy, predictions, true_labels = evaluate_model(model, testLoader)
print(f'Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

# Plot training history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.tight_layout()
plt.show()

print(f"Best validation accuracy: {max(val_accs):.4f}")
print(f"Final test accuracy: {test_accuracy:.4f}")

Data source import complete.
Using device: cpu
Train: 4267 images, Val: 1093 images
Loaded 4267 samples with 67 classes
Loaded 1093 samples with 67 classes
Loaded 1340 samples with 67 classes
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /Users/vishali/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:16<00:00, 6.29MB/s]


Starting training...
